In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vaillant/rsna-pediatric-bone-age-challenge-n1200")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'rsna-pediatric-bone-age-challenge-n1200' dataset.
Path to dataset files: /kaggle/input/rsna-pediatric-bone-age-challenge-n1200


In [4]:
import os

base_path = "/kaggle/input/rsna-pediatric-bone-age-challenge-n1200/"
print(os.listdir(base_path))

['images', 'train.csv']


In [5]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, Concatenate
from tensorflow.keras.models import Model
import pandas as pd
import os

base_path = "/kaggle/input/rsna-pediatric-bone-age-challenge-n1200/"

csv_path = os.path.join(base_path, "train.csv")
img_dir = os.path.join(base_path, "images")

df = pd.read_csv(csv_path)
df['image_path'] = df['pid'].astype(str) + '.png'

print(f"Jadval muvaffaqiyatli yuklandi. Jami rasmlar: {len(df)}")

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

def m_input_generator(generator, subset_name):
    gen = generator.flow_from_dataframe(
        dataframe=df,
        directory=img_dir,
        x_col='image_path',
        y_col=['bone_age', 'female'],
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset=subset_name
    )
    while True:
        data = next(gen)
        images = data[0]
        labels = data[1][:, 0].astype('float32')   # bone_age
        genders = data[1][:, 1].astype('float32')  # female (0.0 yoki 1.0)

        yield {"image_input": images, "gender_input": genders}, labels

train_gen = m_input_generator(datagen, 'training')
val_gen = m_input_generator(datagen, 'validation')

image_input = Input(shape=(224, 224, 3), name='image_input')
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model(image_input)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)

gender_input = Input(shape=(1,), name='gender_input')
y = Dense(16, activation='relu')(gender_input)

combined = Concatenate()([x, y])
combined = Dense(64, activation='relu')(combined)
combined = Dropout(0.2)(combined)

output = Dense(1, activation='linear', name='output')(combined)

model = Model(inputs=[image_input, gender_input], outputs=output)
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

steps_per_epoch = int(len(df) * 0.8) // 32
validation_steps = int(len(df) * 0.2) // 32

print("\nModel rasm va JINS ko'rsatkichini hisobga olgan holda o'qitilmoqda...")
model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=10
)

model.save('suyak_yoshi_gender_model.keras')
print("\nYangi 'suyak_yoshi_gender_model.keras' modeli muvaffaqiyatli saqlandi!")

Jadval muvaffaqiyatli yuklandi. Jami rasmlar: 1200
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Model rasm va JINS ko'rsatkichini hisobga olgan holda o'qitilmoqda...
Found 960 validated image filenames.
Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 102.5515 - mae: 102.5515Found 240 validated image filenames.
30/30 ━━━━━━━━━━━━━━━━━━━━ 60s 2s/step - loss: 75.9155 - mae: 75.9155 - val_loss: 45.7329 - val_mae: 45.7329
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - loss: 37.3633 - mae: 37.3633 - val_loss: 33.2476 - val_mae: 33.2476
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - loss: 33.1764 - mae: 33.1764 - val_loss: 31.4240 - val_mae: 31.4240
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - loss: 30.7963 - mae: 30.7963 - val_loss: 28.8895 - val_mae: 28.8895
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 84s 3s/step - loss: 28.4934 - mae: 28.4934 - val_loss: 26.6421 - val_mae: 26.6421
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - loss: 26.7091 - mae: 26.7091 - va